# Work on the OCSEAN DATA for lingpy modelling

This is incomplete, but contains some very important functionality to find issues with the data with a **check** function to search all languages for symbols or word segments.

At present, it assumes you will rerun "process_linguistic_excel_data.ipynb" if you make modifications to the raw data. This should be streamlined.

In [1]:
import pandas as pd
from lingpy import * # We're just importing everything from lingpy for simplicity
from lingpy.sequence.sound_classes import ipa2tokens
import re
import os
import math

/Users/madjl/uv/ocsean_venv/lib/python3.12/site-packages/pybtex/plugin/__init__.py:26: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


## Metadata

We created a list of "good" languages in process_linguistic_excel_data.ipynb, which we can treat as the main reference to link languages to files.

In [2]:
metadata=pd.read_excel("OCSEAN_initial_englishsheet.xlsx")  
metadata.index=metadata['Language_AsInFile']
metadata.head()

,Unnamed: 0,Language_BasedOnMasterSheet,ISO code (from Original_FileName),COUNTRY,LocalWordsPresent,QC_done,AddedToTheUnitedLanguagesMasterFile,Comment,Separator,Mismatches_Eng,...,Original_FileName,QC_FileName,Google Map Coordinate,latitude,longitude,Has_QC_File,File_Source,Has_IPA,Has_English,Language_AsInFile
Language_AsInFile,,,,,,,,,,,,,,,,,,,,,
Abui_Bunggeta,0,Abui Bunggeta,ABZ,Indonesia,604,Yes,Yes,Some English and or Indonesian words in this l...,", and ; and ~ (?). Only used , and ;",41.0,...,OCSEAN-ABZ_20240605-BUNGGETA_WORDLIST.xlsx,OCSEAN-ABZ_20240605-BUNGGETA_WORDLIST_PostQC.xlsx,NaN,NaN,NaN,True,CleanedFiles-v1.1,True,True,Abui_Bunggeta
Abui_Mobyetang,2,Abui Mobyetang,ABZ,Indonesia,602,Yes,Yes,English Elicitation = English actually. Some E...,", and ; and ~ (?). Only used , and ;",41.0,...,OCSEAN-ABZ_20240610-MOBYETANG_WORDLIST.xlsx,OCSEAN-ABZ_20240610-MOBYETANG_WORDLIST_PostQC....,NaN,NaN,NaN,True,CleanedFiles-v1.1,False,True,Abui_Mobyetang
Abui_Pelman,3,Abui Pelman,ABZ,Indonesia,602,Yes,Yes,English Elicitation = English actually. Some E...,", and ; and ~ (?). Only used , and ;",41.0,...,OCSEAN-ABZ_20240610-PELMAN_WORDLIST.xlsx,OCSEAN-ABZ_20240610-PELMAN_WORDLIST_PostQC.xlsx,NaN,NaN,NaN,True,CleanedFiles,False,True,Abui_Pelman
Agta,9,Agta,AGT,the Phillipines,952,"Yes, but not ideal",Yes,There seems to be an issue with the removal of...,", and /",13.0,...,OCSEAN-AGT_20240124-WORDLIST_1TO1228.xlsx,OCSEAN-AGT_20240124-WORDLIST_1TO1228_PostQC.xlsx,NaN,NaN,NaN,True,CleanedFiles-v1.1,False,True,Agta
Agusan_Manobo,10,Agusan Manobo,MSM,the Phillipines,1225,Yes,Yes,Tagalog and English present and both match the...,",",0.0,...,OCSEAN-MSM_20240517-WORDLIST_1to1228.xlsx,OCSEAN-MSM_20240517-WORDLIST_1to1228_PostQC.xlsx,"8.712999621236902, 125.66373798590386",8.713,125.663738,True,CleanedFiles-v1.1,False,True,Agusan_Manobo


### Using the metadata:

We cal lookup the file associated with 'Balinese' with the following code. Its in the "CleanedFiles" folder.

In [3]:
print(metadata.loc['Balinese','QC_FileName'])
print(metadata.loc['Ata',:])

OCSEAN-BAN_20220714-WORDLIST_PostQC.xlsx
Unnamed: 0                                                                            17
Language_BasedOnMasterSheet                                                          Ata
ISO code (from Original_FileName)                                                    ATM
COUNTRY                                                                  the Phillipines
LocalWordsPresent                                                                    938
QC_done                                                                              Yes
AddedToTheUnitedLanguagesMasterFile                                                  Yes
Comment                                The original file seems to have many other lan...
Separator                                                                              /
Mismatches_Eng                                                                       2.0
Mismatches_IndoOrTaga                                                

## Reading our data

We can read in our data very simply, as its been processed into a simple dataset of 'concept','form' and 'doculect'. (There is an ID column too, just a number to identify each row).

In [4]:
od = pd.read_table('OCSEAN_initial_joineddata.tsv')
od=od[['ID','concept','form','doculect']]
od=od.dropna()
od.shape

(72177, 4)

Its very important to be able to search for "bad characters" that cannot be converted to IPA and might indicate problems for manual examination. These functions do that, and we have examples of their use below. Don't worry too much about how they work just yet.

First we define a list of "bad symbols" for later use, then define the functions.

In [118]:
badsymbols=['(',')', # Brackets
            ' ','\t', # Spaces
            'ė', 'ό', 'ǹ', 'ѐ', # Accented symbols
            '\'','’','´', # quotes used as glottels
            '–','-','=', # separator symbols? NB these are technically different!
            '\\','?','…' # others!
           ]

In [119]:
def checkpair(od,symbol, language,reportok=False,show=False,what='form',regex=False):
    # Check a single symbol/language pair
    tlang=od[od['doculect']==language]
    test=tlang[what].str.contains(symbol, regex=regex)
    total=0
    if (any(test)):
        total += test.sum()
        print(language,"contains",total, 
                      "of the",symbol,"symbol")
        if show:
            showforms=[b for a, b in zip(test,tlang.index) if a]
            for x in showforms:
                print("...",tlang['form'].loc[x]," - concept:", tlang['concept'].loc[x])
    elif reportok:
        print(language, "OK!")
    return(total)
    
def check(od, symbols=badsymbols, languages="all", reportok=False,show=False,what='form',regex=False):
    ## Check for all specified symbols in all requested languages
    if languages == "all":
        languages = list(od['doculect'].unique())
    totals = {}
    for symbol in symbols:
        totals[symbol] = {}
        for language in languages:
            totals[symbol][language] = checkpair(od,symbol,language,reportok,show,what,regex)
    return totals

However, this data still contain forms separated by a comma, which need to be put into separate rows. Here is an example where we search for the comma with the simpler of the two functions:

In [120]:
totals=checkpair(od,',','Agutaynen',show=True)

Agutaynen contains 2 of the , symbol
... laod, kadadaliman  - concept: ocean
... pantay, kapatagan  - concept: path, trail


In [121]:
od.loc[(od['doculect']=='Agutaynen') & (od['concept']=='ocean'),:]

,ID,concept,form,doculect
4591,4591,ocean,"laod, kadadaliman",Agutaynen


In [122]:
test=check(od,what='form', symbols=['*'],show=True)

And here is an example using the more complicated function, which accepts lists as input to check multiple languages and/or symbols.

In [123]:
totals=check(od,symbols=['(v)'],show=True)

In [124]:
total=check(od,symbols=['NOUN'],what='concept',show=True)

In [125]:
total=check(od,symbols=[r'\(\w+\)'],regex=True,show=True)

In [126]:
total=check(od,symbols=[r'\b[A-Z]'],regex=True)

Abui_Bunggeta contains 74 of the \b[A-Z] symbol
Abui_Mobyetang contains 1 of the \b[A-Z] symbol
Abui_Pelman contains 3 of the \b[A-Z] symbol
Akeanon contains 205 of the \b[A-Z] symbol
Boholano contains 2 of the \b[A-Z] symbol
Bontoc contains 1 of the \b[A-Z] symbol
Bulus contains 7 of the \b[A-Z] symbol
Enggano contains 1 of the \b[A-Z] symbol
Hanunuo contains 1 of the \b[A-Z] symbol
Hattang_Kaye contains 4 of the \b[A-Z] symbol
Hiligaynon contains 1 of the \b[A-Z] symbol
Ilognon contains 3 of the \b[A-Z] symbol
Ilokano contains 1 of the \b[A-Z] symbol
Inabaknon contains 1 of the \b[A-Z] symbol
Ivatan_Isabtangen contains 9 of the \b[A-Z] symbol
Ivatan_Ichbayatan contains 5 of the \b[A-Z] symbol
Kamayo contains 1 of the \b[A-Z] symbol
Kapampangan contains 2 of the \b[A-Z] symbol
Kolibogon contains 1 of the \b[A-Z] symbol
Kusa contains 1 of the \b[A-Z] symbol
Manea contains 1 of the \b[A-Z] symbol
Meranaw contains 3 of the \b[A-Z] symbol
Minamanwa contains 2 of the \b[A-Z] symbol
Obo con

In [127]:
#totals=check(od,symbols=['don'],show=True,what='concept')

In [128]:
totals=check(od,symbols=['~'],show=True)

In [129]:
totals=check(od,symbols=['?'],show=True)

Chabacano_Caviteno contains 1 of the ? symbol
... no ba?  - concept: isn't that so?
Ibaloi_Ibaloy contains 11 of the ? symbol
... ay sikato?  - concept: isn't that so?
... kas-ano?  - concept: how?
... sampika?  - concept: how much?
... piga?  - concept: how many?
... nganiman?  - concept: what?
... pigan?  - concept: when?
... shi?  - concept: where?
... tuwad man?  - concept: where?
... sifa?  - concept: who?
... ngantoy?  - concept: why?
... sifa?  - concept: whose?


In [130]:
totals=check(od,symbols=[';'],show=True)

Abui_Bunggeta contains 17 of the ; symbol
... buku; bukuw  - concept: earth
... simooi; smooi  - concept: wind
... hoot; oot  - concept: rainbow
... tawooqa; tawoqa  - concept: earthquake
... ja palaata; ya palaata  - concept: ice
... nakeerang; lakeerang  - concept: snow
... liang; lieng; leeng  - concept: day
... adiki; adik  - concept: mat
... qo; ko  - concept: grass, brush
... tuk; tuuk  - concept: milk
... jeeng ; yeeng  - concept: how much?
... teuda; tuwda  - concept: how?
... yaa; jaa  - concept: go
... ret; rec  - concept: bite
... hasaala naha; hasaala naGa  - concept: innocent
... bei hasaala naha; bei Gasaala naGa  - concept: innocent
... woharaana; woaraana  - concept: stop
Agusan_Manobo contains 1 of the ; symbol
... adiya to; yain lugar  - concept: transfer (another location)
Arta contains 14 of the ; symbol
... tabug; tubog  - concept: mud
... mepipiyaya; kamahalan  - concept: volcano
... laab; bituwa; laab na bituwa  - concept: cave
... linung-ab; dikalabitu  - concep

In [131]:
tlang=od
test=tlang['form'].str.contains('|', regex=False)
any(test)
#check(od,symbols=['&'],show=True)

False

Splitting on a ";" requires a bit more care to be sure - it looks like these might be used as meaningful sounds and should be re-examined?

However, this is how we split on , and ; using the 'explode' function which duplicates rows when there is a list as an entry.

In [132]:
od['form']=od['form'].str.split(',') # replace strings with , with a list of values
od=od.explode('form') # duplicate rows
od['form']=od['form'].str.split(';') # Same for ;
od=od.explode('form') 
od=od.dropna() # Remove anything that has become missing or invalid
od['ID']=range(od.shape[0]) # Give the new index
od.index=od['ID']
print(od.head(15))
print(od.shape)

    ID    concept              form       doculect
ID                                                
0    0        sun             wariy  Abui_Bunggeta
1    1       moon              'uya  Abui_Bunggeta
2    2       star             furiy  Abui_Bunggeta
3    3        sky            'adiiy  Abui_Bunggeta
4    4      earth              buku  Abui_Bunggeta
5    5      earth             bukuw  Abui_Bunggeta
6    6      cloud            taboqi  Abui_Bunggeta
7    7       wind            simooi  Abui_Bunggeta
8    8       wind             smooi  Abui_Bunggeta
9    9       rain             anuui  Abui_Bunggeta
10  10    drizzle  anuui wobiyaanra  Abui_Bunggeta
11  11    drizzle   anuui wobiyaana  Abui_Bunggeta
12  12    drizzle      anuui paawal  Abui_Bunggeta
13  13        dew               moo  Abui_Bunggeta
14  14  mist, fog            taboqi  Abui_Bunggeta
(72301, 4)


## Working towards IPA

We need our symbols to be readable by **lingpy**, and to use a comparable way of encoding meaningful phonetic variation to "tokens", i.e. fundamental sounds.

Any string can be coerced to IPA using `ipa2tokens` assuming that it doesn't contain symbols forbidden in IPA. To convert a string such as "wobiyaanra" to such tokens we can call `ipa2tokens`. But this returns the tokens as a list, not as a string, so we've made a function to do this for every string in a list, and return a list.

This is a test, because it will fail on forbidden symbols. (Try introducing a space into the example...)

In [133]:
def ipa2tokens_list(d):
    ## Takes a list of dataframe column
    ## Apply transformation to words that are present, and collapse that back to a string
    ret = [''.join(ipa2tokens(x)) for x in d]
    return ret

In [134]:
ipa2tokens("anuui.wobiyaana") # Takes only a string, returns a list

['a', 'n', 'uui', 'w', 'o', 'b', 'iyaa', 'n', 'a']

In [135]:
ipa2tokens_list(["anuui.wobiyaana","simooi"]) # takes a list, returns a list

['anuuiwobiyaana', 'simooi']

In [136]:
ipa2tokens("ka-yog")

['k', 'a', 'yo', 'g']

In [137]:
ipa2tokens("kayog")

['k', 'ayo', 'g']

In [138]:
ipa2tokens("ka.yog")

['k', 'a', 'yo', 'g']

In [139]:
od.loc[4270:4275,:] # This was once a split 'Agutaynen' 'ocean'

,ID,concept,form,doculect
ID,,,,
4270,4270,ninety,kasiyaman,Agusan_Manobo
4271,4271,one hundred,isang gatus,Agusan_Manobo
4272,4272,one hundred and twenty three,isang gatus ug kawha'an nga usik,Agusan_Manobo
4273,4273,two hundred,duha ka gatus,Agusan_Manobo
4274,4274,one hundred thousand,isang gatus nga libo,Agusan_Manobo
4275,4275,thousand,libo,Agusan_Manobo


# Some testing

We will use the check function on different symbols. These should be manually fixed, unless an automatic rule is safe and appropriate. 

In [140]:
## This is how I got the "ė" symbol
od.loc[(od['doculect']=='Enggano') & (od['concept']=='dog'),:]

,ID,concept,form,doculect
ID,,,,
26020,26020,dog,bė,Enggano


In [141]:
test=check(od,what="concept",symbols=['firstbornchild'],show=True)

Akeanon contains 2 of the firstbornchild symbol
... Panganay  - concept: firstbornchild
... kamug-eangan  - concept: firstbornchild
Ati contains 1 of the firstbornchild symbol
... subang  - concept: firstbornchild
Kinaray_a contains 1 of the firstbornchild symbol
... panganay  - concept: firstbornchild
Tausug contains 1 of the firstbornchild symbol
... kakah  - concept: firstbornchild


In [142]:
totals=check(od,symbols=['ẽ'])
totals=check(od,symbols=['á']) # Not present
totals=check(od,symbols=['à'])
totals=check(od,symbols=['ė'])
totals=check(od,symbols=['ǹ'])
totals=check(od,symbols=['ň'])

Balinese contains 107 of the ẽ symbol
Agusan_Manobo contains 5 of the à symbol
Tboli contains 21 of the à symbol
Enggano contains 23 of the ė symbol
Ivatan_Ichbayatan contains 40 of the ǹ symbol
Ivatan_Isabtangen contains 26 of the ň symbol


In [143]:
totals=check(od,languages=['Agusan_Manobo'], symbols=['à'],show=True)

Agusan_Manobo contains 5 of the à symbol
... nanà  - concept: pus
... lagà  - concept: boil
... kalà  - concept: frying pan
... pinatà  - concept: bamboo water container
... batà te manok  - concept: egg


We have identified that 'ngh' needs treating carefully to separate out 'ng.h' from 'n.gh'. Both 'ng' and 'gh' might then be needing an IPA symbol.

In [144]:
totals=check(od,symbols=['ngh'],show=True)

Abui_Mobyetang contains 1 of the ngh symbol
... minghede  - concept: surrender
Agta contains 1 of the ngh symbol
... tanghali  - concept: noon, midday
Agusan_Manobo contains 4 of the ngh symbol
... ugpanghihingutohay  - concept: delouse, remove lice
... dalanghita  - concept: citrus
... ma'anghang  - concept: spicy
... estranghero  - concept: stranger
Akeanon contains 10 of the ngh symbol
... kamanghuran  - concept: youngest child
... Manghod  - concept: brother
... Manghod  - concept: sister
... Manghod nga eaki  - concept: youngerbrother
... Manghod nga bayi  - concept: younger sister
... manghod  - concept: youngersibling
... balinghoy  - concept: cassava
... anghit  - concept: smell
... panghi  - concept: smell
... Estranghero  - concept: stranger
Arta contains 2 of the ngh symbol
... buwanghina  - concept: rough bamboo, giant bambooo (dendrocalamus asper)
... manghuras  - concept: wash
Ati contains 1 of the ngh symbol
... panghabok habok kay mata  - concept: conjunctivitis
Balanga

In [145]:
totals=check(od,symbols=['gh'],languages=["Waray"],show=True)

Waray contains 11 of the gh symbol
... dughan  - concept: chest
... pakighilawas  - concept: sexual intercourse, copulate
... koghad  - concept: phlegm
... naghagong  - concept: snore
... paghigop  - concept: sip, slurp
... paghukay  - concept: dig
... dalanghita  - concept: citrus
... balinghoy  - concept: cassava
... daghan  - concept: many
... estranghero  - concept: stranger
... pakighilawas  - concept: adultery


In [146]:
totals=check(od,symbols=['('],show=True)

# Checking everything

Here we check for every symbol in our list. It makes a very long output!

In [147]:
totals=check(od)

Abui_Bunggeta contains 353 of the   symbol
Abui_Mobyetang contains 248 of the   symbol
Abui_Pelman contains 209 of the   symbol
Agta contains 80 of the   symbol
Agusan_Manobo contains 101 of the   symbol
Agutaynen contains 135 of the   symbol
Akeanon contains 70 of the   symbol
Arta contains 185 of the   symbol
Ata contains 240 of the   symbol
Ati contains 108 of the   symbol
Ba_a contains 341 of the   symbol
Balangao contains 152 of the   symbol
Bali_Aga contains 34 of the   symbol
Balinese contains 174 of the   symbol
Batak contains 111 of the   symbol
Bicolano contains 141 of the   symbol
Boholano contains 135 of the   symbol
Bolinao contains 84 of the   symbol
Bontoc contains 105 of the   symbol
Buhid contains 37 of the   symbol
Bulus contains 165 of the   symbol
Chabacano_Caviteno contains 77 of the   symbol
Cuyunon contains 100 of the   symbol
Dela contains 286 of the   symbol
Gaddang contains 117 of the   symbol
Hanunuo contains 65 of the   symbol
Hattang_Kaye contains 278 of th

## Fixing the list

Here are some options for fixing the list. Carefully check each choice and see if it is safe. If not, we need to go back to the original data and edit it.

In [148]:
total=check(od,symbols=['escort'],what='concept',show=True)

Agusan_Manobo contains 1 of the escort symbol
... ug-duma  - concept: escort, bring s.o. somewhere
Agutaynen contains 2 of the escort symbol
... hatid  - concept: escort, bring s.o. somewhere
... henated  - concept: escort, bring s.o. somewhere
Akeanon contains 2 of the escort symbol
... Ibhan  - concept: escort, bring
s.o. somewhere
... ihatod  - concept: escort, bring
s.o. somewhere
Arta contains 1 of the escort symbol
... itugan  - concept: escort, bring s.o. somewhere
Ata contains 1 of the escort symbol
... pog bantoy  - concept: escort, bring s.o. somewhere
Ba_a contains 1 of the escort symbol
... ninin  - concept: escort, bring s.o. somewhere
Balangao contains 1 of the escort symbol
... itnod  - concept: escort, bring s.o. somewhere
Balinese contains 1 of the escort symbol
... ateh  - concept: escort, bring s.o. somewhere
Batak contains 1 of the escort symbol
... iated  - concept: escort, bring s.o somewhere
Bicolano contains 1 of the escort symbol
... ihatod  - concept: escort, 

In [149]:
total=check(od,symbols=['x'])
total=check(od,symbols=['x'],languages=['Ata','Chabacano_Caviteno','Kolibogon','Minamanwa'],show=True)

Chabacano_Caviteno contains 1 of the x symbol
Ivatan_Ichbayatan contains 200 of the x symbol
Kolibogon contains 1 of the x symbol
Chabacano_Caviteno contains 1 of the x symbol
... extende  - concept: spread out
Kolibogon contains 1 of the x symbol
... taga x  - concept: has x


In [150]:
#total=check(od,symbols=['y'],languages=['Ba_a'],show=True)
total=check(od,symbols=['ny'],languages=['Balinese'],show=True)

Balinese contains 56 of the ny symbol
... nyanyad  - concept: mud
... nyat  - concept: lowtide
... nyama muani  - concept: brother
... nyama luh  - concept: sister
... nyawan  - concept: bee
... penyu  - concept: turtle
... nyilapin  - concept: lick
... nyonyo  - concept: breast
... tutuk nyonyo  - concept: nipple, teat
... nyonyo  - concept: milk
... nyilem  - concept: drown
... kenyel  - concept: tired, weary
... punyah  - concept: drunk, intoxicated
... nyepsep  - concept: suck, breastfeed
... kinyukan  - concept: chew
... penyemeng  - concept: breakfast
... nyuwahin kutu  - concept: comb for lice with a fine tooth comb
... nyemak yẽh  - concept: fetch water
... nyejek padi  - concept: thresh
... nyiu  - concept: winnowing basket
... punyan kayu  - concept: stem, trunk
... punyan  - concept: tree
... punyan ental  - concept: lontar palm
... punyan cemara  - concept: casuarina tree
... nyuh  - concept: coconut
... nyuh  - concept: old coconut
... punyan bingin  - concept: banyan tree

In [151]:
ipa2tokens('nyupir')

['n', 'yu', 'p', 'i', 'r']

In [152]:
## Brackets - throw away everything!
od['form'] = od['form'].str.split('(').str[0]
## Symbolsipa2tokens("kayog")
od['form']=[x.replace('–','.') for x in od['form']] # long dash should be a short pause
od['form']=[x.replace('-','.') for x in od['form']] # short dash should be a short pause
od['form']=[x.replace('=','') for x in od['form']] # equals
od['form']=[x.replace(')','') for x in od['form']] # end brackets - shouldn't be here, but are
od['form']=[x.replace('\'', 'ʔ') for x in od['form']] # the ' character, glottal stop?. This is special in Python so we've "escaped" it here with a \, to match the actual character
od['form']=[x.replace('’', 'ʔ') for x in od['form']] # all the different types of quote symbol- glottal stop?
od['form']=[x.replace('´', 'ʔ') for x in od['form']] # all the different types of quote symbol- glottal stop?
od['form']=[x.replace('‘', 'ʔ') for x in od['form']] # all the different types of quote symbol- glottal stop?
od['form']=[x.replace('*', '') for x in od['form']] # question mark
od['form']=[x.replace('?', '') for x in od['form']] # question mark
od['form']=[x.replace('\\', '') for x in od['form']] # the backslash symbol. This could be encoding something like IPA?
## Phonetics
od['form']=[x.replace('ė', 'e') for x in od['form']] # Doesn't match
### od['form']=[x.replace(u"\u0113", 'ɛ') for x in od['form']] # Doesn't match
od['form']=[x.replace('ό', 'ɔ̝') for x in od['form']]
od['form']=[x.replace('ǹ', 'n̪') for x in od['form']]
od['form']=[x.replace('ѐ', 'e') for x in od['form']] # The IPA should be /e/ but that isn't single character?
od['form']=[x.replace('ň', 'ŋ') for x in od['form']] # 
od['form']=[x.replace('ng', 'ŋ') for x in od['form']] # The IPA should be /e/ but that isn't single character?
od['form']=[x.replace('x', 'h') for x in od['form']] # Correct for Ivatan, but weird in other contexts! Not many 'x' appear
# Cases, numbers, spaces
od['form']=[x.replace('…', '') for x in od['form']]
od['form']=[x.lower() for x in od['form']]
od['form']=[re.sub(r'[\d]+', '', x) for x in od['form']]
od['form']=[re.sub(r'[\s]+', '.', x) for x in od['form']]
od = od[od['form'] != '']
od = od[od['form'] != '.']
##od.loc[:,'ID']=od.index+1

In [153]:
total=check(od,symbols=['gh'],show=True)

Agusan_Manobo contains 11 of the gh symbol
... mataghum  - concept: snow
... paghindo  - concept: point at
... ughugok  - concept: snore
... ughinoktok  - concept: mourn, be in mourning
... ughimatoy  - concept: kill
... ughiŋyawon  - concept: fever
... ughihiŋyawon  - concept: cold, influenza
... ughupoŋ  - concept: swell
... aghod  - concept: broom
... daghan  - concept: many
... ug.panaghoy  - concept: whistle
Akeanon contains 3 of the gh symbol
... dughan  - concept: chest
... dighay  - concept: burp, belch
... paghigugma  - concept: love
Balangao contains 4 of the gh symbol
... ugha  - concept: deer
... heghegʔke  - concept: hiccough
... amgn.hegheg.ke  - concept: thresh
... cheghen  - concept: full
Batak contains 2 of the gh symbol
... bugho  - concept: jealous
... pagtaghoy  - concept: whistle
Bicolano contains 12 of the gh symbol
... daghan  - concept: chest
... maghibarmas  - concept: shave
... maghabol  - concept: sew
... maghasok  - concept: sow
... sighin  - concept: broom


In [154]:
#total=check(od,reportok=True)

### Bug hunting

When there were bad symbols left, 'ipa2tokens_list' would give an error which was sometimes hard to debug. It was helpful to split the data into blocks of 100 and report which still failed.

You can ignore all of this now, its kept for posterity.

In [155]:
## Hunting for bad sequences that cannot be converted to IPA
## Print out the "batch number" of a block of 100 that break the algorithm
for i in range(math.floor(len(od)/100)):
    try:
        tmp=ipa2tokens_list(od['form'][(100*i):(100*i+100)])
    except:
        print(i)

In [156]:
## How we investigate a "bad" batch.
i=255
for j in range(2100*i,100*i+100):
    print(od['form'][j]) 
print(list(od['form'][(100*i):(100*i+100)]))
tmp=ipa2tokens_list(od['form'][(100*i):(100*i+100)])

['sia.dea', 'ia.rae', 'sia.ata', 'seri', 'ia.sie', 'naa.e', 'ia', 'naa', 'mamana', 'mbeda', 'endo', 'beu.tee', 'suŋgu.naboboa', 'ri.rii', 'nasaraii', 'leo.', '.nahani', 'lao.hela', 'nahani', 'soru', 'buka', 'ena.', '.tatana', 'ena', 'tatana', 'kekeh', 'naru', 'naru', 'eeku', 'suu', 'sia.talada', 'ona', 'dii', 'losaa', 'doo', 'rulu', 'muri', 'dii', 'ona', 'mori', 'haŋga', 'monae', 'anak', 'naru', 'boboku', 'eeku', 'eeku', 'loa', 'makabia', 'fau', 'niis', 'ŋgoda', 'roma', 'kabao', 'ndoos', 'buku', 'lolonda', 'boboŋgo', 'ndola', 'esa', 'rua', 'telu', 'haa', 'lima', 'nee', 'hitu', 'falu', 'sio', 'sanahulu', 'sanahulu.esa', 'sanahulu.rua', 'sanahulu.telu', 'sanahulu.haa', 'sanahulu.lima', 'sanahulu.nee', 'sanahulu.hitu', 'sanahulu.falu', 'sanahulu.sio', 'rua.nulu', 'rua.nulu.esa', 'rua.nulu.rua', 'rua.nulu.telu', 'rua.nulu.haa', 'rua.nulu.lima', 'telu.nulu', 'haa.nulu', 'lima.nulu', 'nee.nulu', 'hitu.nulu', 'falu.nulu', 'sio.nulu', 'natun.esa', 'natun.esa.rua.mulu.telu', 'natun.rua', 'rifon

In [157]:
# Convert to IPA
od['ipa']=ipa2tokens_list(od['form'])
od['doculect']=[re.sub(r"\s+", '_', x) for x in od['doculect']]
od['concept']=[re.sub(r"\s+", '_', x) for x in od['concept']]
od.to_csv('OCSEAN_processed_joineddata.tsv',sep='\t')

In [158]:
test=check(od,symbols=['*'],show=True)

# Running pyling and models within it

We're now ready to run our data through models!

Its easy enougfh now: just read the wordlist in, run 'LexStat' to set up some basic details, and then run 'lexstat'. However that is a bit slow so we've put it in a script that can be run separately, or avoided and we just read the results.

In [5]:
ocseanwl = Wordlist('OCSEAN_processed_joineddata.tsv')

In [6]:
from tabulate import tabulate

In [7]:
print(tabulate([ocseanwl[idx] for idx in range(1, 5)]))

-  -----  ------  -------------  ------
1  moon   ʔuya    Abui_Bunggeta  ʔuya
2  star   furiy   Abui_Bunggeta  furiy
3  sky    ʔadiiy  Abui_Bunggeta  ʔadiiy
4  earth  buku    Abui_Bunggeta  buku
-  -----  ------  -------------  ------


In [8]:
ocseanlex = LexStat(ocseanwl,check=True)

2025-06-23 15:54:17,602 [INFO] No obvious errors found in the data.


KeyboardInterrupt: 

In [ ]:
print(tabulate([ocseanlex[idx] for idx in range(1, 5)]))

In [113]:
from lingpy.compare.sanity import average_coverage, mutual_coverage_check, synonymy 
# Check mutual coverage # Very slow for all languages
#for i in range(10, 0, -1):
#    print(i)
#    if mutual_coverage_check(ocseanlex, i):
#        print("Minimal mutual coverage is at {0} concept pairs.".format(i))
#        break
print('Avarage coverage is {0:.2f}.'.format(average_coverage(ocseanlex)))

Avarage coverage is 0.28.


In [ ]:
synonyms = synonymy(ocseanlex)
# synonyms # This is just a count foe each concept

In [ ]:
num_synonyms = len(ocseanlex) - len(synonyms)
syn_ratio = 1 - (len(synonyms)/len(ocseanlex))
if num_synonyms == 0:
    print('Found {0} potential synonyms.'.format(num_synonyms))
else:
    print('Found {0} potential synonyms ({1:.2}%):'.format(num_synonyms, syn_ratio*100.0))
    for (language, concept), count in sorted(synonyms.items(), key=lambda x: x[0][0]):
        if count > 10:
            print('{0:15}  {1:12}  {2}'.format(language, concept, count))

## Cognate detection

We will now detect cognates.

The following uses a "script" (.py file) that we can run separately. This allows us to:

a) capture the output (it gets messy!)
b) streamline the processing by running it once and saving the output.

The next line (`%pycat lexstat_ocsean.py`) simply prints the contents of the script to the screen, so we can see what we are doing.

The following line (`!python lexstat_ocsean.py`) runs the same script. The `&>` is called "shell redirection" and captures all the "printed" statements and warnings and puts them into the `.log` file.

Uncomment the line after to rerun cognate detection. Lots of details can be changed.

In [ ]:
%pycat lexstat_ocsean.py
## Note that 'pycat' is a feature of "Jupyter notebook" which is the interface we are using, not of python itself.

In [ ]:
!python lexstat_ocsean.py &> lexstat_ocsean.log

In [10]:
lexdf=pd.read_table(os.path.join('data','OCSEAN_processed_cognatedetected.tsv'),skiprows=2,comment='#')
print(lexdf.shape)
lexdf.head()

(69890, 15)


,ID,ID.1,CONCEPT,FORM,DOCULECT,IPA,TOKENS,SONARS,PROSTRINGS,CLASSES,LANGID,NUMBERS,WEIGHTS,DUPLICATES,COGID
0,49095,49095,(coral)_ree,fet.siaf.kenan,Mawes_Wares,fetsiafkenan,f e t s ia f k e n a n,3 7 1 3 7 3 1 7 4 7 4,AXBCYMBYBYN,BETSIBKENAN,46,46.B.C 46.E.V 46.T.C 46.S.C 46.I.V 46.B.c 46.K...,2.0 1.5 1.75 1.5 1.3 1.1 1.75 1.3 1.75 1.3 0.8,0,1
1,57,57,(coral)_reef,koqai,Abui_Bunggeta,koqai,k o q ai,1 7 1 7,AXBZ,KUKA,1,1.K.C 1.U.V 1.K.C 1.A.V,2.0 1.5 1.75 0.8,1,2
2,1057,1057,(coral)_reef,tama.kokai,Abui_Mobyetang,tamakokai,t a m a k o k ai,1 7 4 7 1 7 1 7,AXBYBYBZ,TAMAKUKA,2,2.T.C 2.A.V 2.M.C 2.A.V 2.K.C 2.U.V 2.K.C 2.A.V,2.0 1.5 1.75 1.3 1.75 1.3 1.75 0.8,0,2
3,1732,1732,(coral)_reef,tama.kokai,Abui_Pelman,tamakokai,t a m a k o k ai,1 7 4 7 1 7 1 7,AXBYBYBZ,TAMAKUKA,3,3.T.C 3.A.V 3.M.C 3.A.V 3.K.C 3.U.V 3.K.C 3.A.V,2.0 1.5 1.75 1.3 1.75 1.3 1.75 0.8,0,2
4,2366,2366,(coral)_reef,bahura,Agusan_Manobo,bahura,b a h u r a,1 7 3 7 5 7,AXBYBZ,PAHYRA,4,4.P.C 4.A.V 4.H.C 4.Y.V 4.R.C 4.A.V,2.0 1.5 1.75 1.3 1.75 0.8,0,3


In [ ]:
from lingpy import Alignments
ocseanlexproc = LexStat(os.path.join('data','OCSEAN_processed_cognatedetected.tsv'),check=True)
alms = Alignments(ocseanlexproc, transcription='form', ref="cogid", segments="segments")
alms.align(method='library')

In [ ]:
from lingpy import SCA
print(SCA(alms.msa['cogid'][2]))

In [ ]:
dists = alms.get_distances(ref='cogid')
alms.output('dst', filename='OCSEAN_processed_distances')

In [ ]:
tdists=pd.read_table('OCSEAN_processed_distances.dst',skiprows=1,header=None)
theaders = list(tdists.iloc[:,0])
dists  = pd.DataFrame(tdists.values[:,1:], columns=theaders,index=theaders)
dists = dists[dists.columns].astype(float)  # or int
dists.iloc[0:10,0:10]

In [ ]:
from lingpy import Tree
tree = Tree(alms.get_tree(ref='cogid', tree_calc='upgma', force=True))
print(tree.asciiArt(compact=True))
nex = write_nexus(wl, mode="BEAST", filename="lieberherrkhobwa-beast.nex")

In [ ]:
from lingpy.convert.cldf import to_cldf
to_cldf(alms, path='ocsean-cldf-new')

In [ ]:
import pandas as pd
import numpy as np
from itertools import combinations

# Get unique list of languages
mylanguages = sorted(lexdf['DOCULECT'].unique())
lang_idx = {lang: i for i, lang in enumerate(mylanguages)}

# Initialize matrix
matrix = np.zeros((len(mylanguages), len(mylanguages)), dtype=int)

# Group by cognate set
for cogid, group in lexdf.groupby('COGID'):
    langs = group['DOCULECT'].unique()
    for lang1, lang2 in combinations(langs, 2):
        i, j = lang_idx[lang1], lang_idx[lang2]
        matrix[i][j] += 1
        matrix[j][i] += 1  # Because it's undirected

# Optional: fill diagonal with how many unique cognate sets each language has
for lang in mylanguages:
    matrix[lang_idx[lang]][lang_idx[lang]] = lexdf[lexdf['DOCULECT'] == lang]['COGID'].nunique()

# Turn into a pandas DataFrame
adj_df = pd.DataFrame(matrix, index=mylanguages, columns=mylanguages)
adj_df.head()
# Show or save
adj_df.to_csv(os.path.join('data','OCSEAN_cognate_adjacency_matrix.csv'))

In [ ]:
# Assuming adj_df is a symmetric matrix
diagonal = np.diag(adj_df)
scaling_factors = np.sqrt(np.outer(diagonal, diagonal))

# Element-wise division
scaled_df = adj_df / scaling_factors

# Convert back to DataFrame with same labels
scaled_df = pd.DataFrame(scaled_df, index=adj_df.index, columns=adj_df.columns)
scaled_df.to_csv(os.path.join('data','OCSEAN_scaled_cognate_adjacency_matrix.csv'))
scaled_df.iloc[0:10,0:10]

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
adj_log = np.log10(scaled_df+0.01)
flat = adj_log.values.flatten()
scaled = StandardScaler().fit_transform(flat.reshape(-1, 1)).reshape(adj_df.shape)

# Convert back to DataFrame
adj_scaled_df = pd.DataFrame(scaled, index=adj_df.index, columns=adj_df.columns)


In [ ]:
scaled_dists = dists

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

mask = np.eye(len(scaled_dists), dtype=bool)

# Create a clustered heatmap
sns.clustermap(
    scaled_dists,
    cmap="coolwarm",          # or any other colormap you like
    linewidths=0.5,
    mask=mask, 
    figsize=(15, 15),
    xticklabels=True,
    yticklabels=True,
    row_cluster=True,
    col_cluster=True,
    cbar_kws={'label': 'Value'},
    dendrogram_ratio=(0.2, 0.2),  # size of dendrograms
    metric="euclidean",           # distance metric for clustering
    method="average"              # linkage method
)

plt.suptitle("Cognate Sharing Visualised", y=1.02)
plt.savefig('CognateSharingHeatmap.png')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

In [ ]:
# Remove the index/column labels for PCA
X = scaled_dists.values

# Standardize the data
X_scaled = StandardScaler().fit_transform(X)

In [ ]:
ncomp=15
pca = PCA(n_components=ncomp)
components = pca.fit_transform(X_scaled)
# Explained variance ratios
explained_variance = pca.explained_variance_ratio_

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(range(1, ncomp+1), explained_variance, marker='o', linestyle='--')
plt.xticks(range(1, ncomp+1))
plt.xlabel('Principal Component')
plt.ylabel('Explained Variance Ratio')
plt.title('Elbow Plot for PCA')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 6))

# Plot only the first two PCs
for i, lang in enumerate(adj_df.index):
    x, y = components[i, 0], components[i, 2]
    plt.scatter(x, y, label=lang)
    plt.text(x + 0.02, y + 0.02, lang, fontsize=9)

plt.title("PCA of Phonemic Distance Matrix (PC1 vs PC2)")
plt.xlabel("PC1 ({:.1f}%)".format(pca.explained_variance_ratio_[0] * 100))
plt.ylabel("PC2 ({:.1f}%)".format(pca.explained_variance_ratio_[1] * 100))
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
collectionsheet=pd.read_excel("OCSEAN_initial_englishsheet.xlsx")
collectionsheet.head()

In [ ]:
from shapely.geometry import Point # Allows ployying points with geopandas and rxr
import geopandas as gpd

languages=collectionsheet[collectionsheet["latitude"].notnull()].copy()
# Convert to GeoDataFrame
languages["geometry"] = languages.apply(lambda row: Point(row["longitude"], row["latitude"]), axis=1)
languages_gdf = gpd.GeoDataFrame(languages, geometry="geometry", crs="EPSG:4326")
languages_gdf["Latitude"]=languages_gdf["latitude"]
languages_gdf["Longitude"]=languages_gdf["longitude"]
languages_gdf["Language"]=languages_gdf["Language_BasedOnMasterSheet"]
#languages_gdf = languages_gdf.to_crs(rds.rio.crs)
languages_gdf.head()

In [ ]:
def date_line_remove (geo,long="Longitude"):
    geo["Longitude2"]=[(x-360 if x>0 else x) for x in geo[long]]
    return geo

languages_gdf=date_line_remove(languages_gdf)
languages_gdf.head()

In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# Assuming polynesia_geo has columns: 'Language', 'Latitude', 'Longitude'

fig = plt.figure(figsize=(14, 10))
ax = plt.axes(projection=ccrs.PlateCarree(central_longitude=180))

# Set map extent to roughly cover the Pacific region
#ax.set_extent([100, -100, 100, 30], crs=ccrs.PlateCarree())
ax.set_extent([100, 180, -30, 30], crs=ccrs.PlateCarree())

# Add map features for context
ax.add_feature(cfeature.LAND, facecolor='lightgray')
ax.add_feature(cfeature.OCEAN, facecolor='lightblue')
ax.add_feature(cfeature.COASTLINE)
ax.add_feature(cfeature.BORDERS, linestyle=':')

# Plot each language point and label it
for idx, row in languages_gdf.iterrows():
    ax.plot(row['Longitude'], row['Latitude'], 'o', color='red', markersize=5, transform=ccrs.PlateCarree())
    ax.text(row['Longitude'] + 1, row['Latitude'] + 1, row['Language'], fontsize=8, transform=ccrs.PlateCarree())

plt.title("Pacific Languages Map", fontsize=16)
plt.show()